This notebook shows how to use BQSKit to compile ensemble of approximations.
This feature is still under development, and will be released soon. This 
tutorial already assumes a basic understanding of BQSKit compilation and 
instantiation-based synthesis. Users who are unfamiliar with BQSKit should start
with (insert jupyter notebooks).

Prerequisites: cvxopt

$ python -m pip install cvxopt

## Contents

1. Introduction to Ensemble Approximation
2. Step by Step Compilation
Partitioning the Circuit
Generating an initial set of solutions
Diversifying our solutions
Solving the sampling distribution
Compiling the Full Ensemble

3. Running the Ensemble
Calculating an Observable
Mitigating Distance over multiple Samples

## Introduction

We are assuming the user has an initial introduction to BQSKit and quantum 
compilation. At a high level, the goal of any quantum compiler to maximize the 
"performance" of a quantum algorithm on a given hardware. In today's NISQ era,
"performance" is usually defined as algorithmic fidelity (fidelity of the 
output state).  As today's HW errors are dominated by 2-qubit gate errors, we 
can measure the quality of compilation by comparing the number of 2-qubit gates.

BQSKit uses instantiation-based compilation to generate circuits that are 
approximately equal to the input algorithm. Because this compilation is able to
reduce the number of 2-qubit gates, the introduction of approximation error
is balanced by the reduction in HW error.

We can increase the approximation error budget passed to BQSKit, which is then
able to find circuits with fewer and fewer 2-qubit gates. However, there comes 
a point at which this trade off between approximation and HW error is no longer
warranted. 

Our technique ouputs a set (ensemble) of these high-approximation solutions.
Together, these circuits form a quantum channel whose approximation error is
quadratically reduced. This allows us to minimize both approximation and HW
error!

In [1]:
# from bqskit.ir import Circuit
# from bqskit.compiler import Compiler

# # Read in a QASM file
# circuit = Circuit.from_qasm_file('path/to/your/file.qasm')

# compiler = Compiler()

# # Now, compile the ensemble. Instead of an output circuit,
# # the workflow generates a CircuitSampler object.
# circuit_sampler = compiler.compile_ensemble(circuit, 
#                                             approximation_distance=1e-2)

# # We will talk about running the circuit sampler in the Section 3

## Step by Step Compilation

In this Section, we will walk through each major step of the workflow and 
explain what is happening under the hood. At a high level we are:

1. Using BQSKit to create high quality (fewer 2-qubit gates) compilations at a
larger approximation distance (epsilon).
2. Diversify the solutions from BQSKit to form a convex hull around the input
unitary. From this paper, given we have a convex hull, there exists a 
probability distribution by which we can sample our set of solutions to 
quadratically reduce the approximation error.
3. Solve for the ideal probability distribution using a Quadratic Solver.
4. Verify that we have formed a convex set.
5. Recombine our block ensembles to form an ensemble over the entire circuit.

As in all of our compilation techniques, we start by partitioning the circuit
to scale our workflow. For this use case, we have found that we find 
*much better* results using larger block sizes, specifically block size 4 and 
above. While this does impact the compilation time, using a larger block size
improves our ability to form a convex hull around the target unitary.

In [1]:
from bqskit.ir import Circuit
from bqskit.compiler import Compiler
from bqskit.passes import ScanPartitioner, UpdateDataPass

input_circ = Circuit.from_file('qae_7.qasm')

# Create a partitioned circuit
# When using bqskit `compile`, we use UpdateData to tell the compiler we
# want to compile an ensemble of solutions 

# We are using block size 4 for better results. This will lead to longer run
# times
with Compiler() as compiler:
    partitioned_circ, data = compiler.compile(input_circ, 
                                        [ScanPartitioner(4)], 
                                        request_data=True)

num_partitions = partitioned_circ.num_operations
print("Split into ", num_partitions, " partitions.")

Split into  6  partitions.


### Generating our initial approximate solutions

In the normal BQSKit workflow we use LEAP, a heurstic-search based algorithm 
that leverages instantiation to find a minimal approximate circuit. For our 
workflow, we extend LEAP to output many solutions that fall within our error
budget.

In [ ]:
from bqskit.passes import LEAPSynthesisPass, ForEachBlockPass
import logging

# Synthesize each block with LEAP. Since we are compiling for an ensemble,
# LEAP will output
with Compiler(runtime_log_level=logging.ERROR) as compiler:
    partitioned_circ, data = compiler.compile(
        partitioned_circ,
        [ForEachBlockPass([
            UpdateDataPass("run_ensemble", True),
            UpdateDataPass("ensemble_name", "qae7"),
            LEAPSynthesisPass(
                success_threshold=1e-3,
                max_solutions=10,
                max_layer=-1
            ),
        ])],
        data=data,
        request_data=True
    )

all_block_data = data[ForEachBlockPass.key][-1]

for i, b_data in enumerate(all_block_data):
    num_solutions = len(b_data["ensemble_circuits"])
    print(f"Block {i} has {num_solutions} approximate solutions")

Block 0 has 12 approximate solutions
Block 1 has 10 approximate solutions
Block 2 has 1 approximate solutions
Block 3 has 2 approximate solutions
Block 4 has 1 approximate solutions
Block 5 has 1 approximate solutions


### Diversifying our solutions

Importantly, the solutions outputted by BQSKit is *not* convex. In 
general, it is hard to generate a set of approximations that are convex and
minimize the number of resources. Our approach works since we have a scalable
method of verifying convexity. This means, we can simply use the blocks that 
have convex approximations in our final ensemble.

Our goal in the next step is to maximize the probability of creating a convex 
set of approximations. From our initial set of solutions, we introduce 
perturbations to spread apart our circuits. This also increase the number of 
circuits we generate per block.

In [7]:
from bqskit.passes import DiversifyEnsemblePass

# Diversify the ensemble of solutions
with Compiler(runtime_log_level=logging.ERROR) as compiler:
    partitioned_circ, data = compiler.compile(
        partitioned_circ,
        [DiversifyEnsemblePass(success_threshold=2e-3)],
        data=data,
        request_data=True
    )

for i, b_data in enumerate(all_block_data):
    num_base_circuits = len(b_data["ensemble_circuits"])
    # How many "noisy" circuits we generate per base circuit from LEAP
    num_params_per_circ = b_data["ensemble_params"][0].shape[0]
    num_solutions = num_base_circuits * num_params_per_circ
    print(f"Block {i} now has {num_solutions} approximate solutions")

RuntimeError: Server connection unexpectedly closed.

### Calculating the Ideal Distribution

At this point, we hope to have created a convex ensemble. Now, we must solve the 
ideal distribution by which to sample this set. This is done using an iterative
quadratic solver.

In [ ]:
from bqskit.passes import GenerateProbabilityPass
import numpy as np

partitioned_circ, data = compiler.compile(
    partitioned_circ,
    [GenerateProbabilityPass()],
    data=data,
    request_data=True
)

for i, b_data in enumerate(all_block_data):
    probs = b_data["ensemble_probabilities"]
    # Find out how many probabilities are above a certain threshold
    threshold = 1e-5
    num_above_threshold = np.sum(probs > threshold)
    print(f"Block {i} has {num_above_threshold} non-zero probabilities")

### Filtering out Bad Ensembles

Now that we have maximized the convexity of our approximate solutions and 
calculated the ideal distribution, we must now answer: "Is our solution set 
convex?"

We have shown (paper to be released) that by calculating the bias of our
ensemble, we can *verify* the convexity of the ensemble in expectation. If our
ensemble is convex, we can prove that the *diamond distance* is quadratically
reduced.

In [ ]:
from bqskit.passes import BiasFilterPass

partitioned_circ, data = compiler.compile(
    partitioned_circ,
    [BiasFilterPass()],
    data=data,
    request_data=True
)

for i, b_data in enumerate(all_block_data):
    # If ensemble circuits is still in data, we have succesfully
    # generated a convex ensemble
    if "ensemble_circuits" not in b_data:
        print(f"Block {i} was filtered out. Defaulting to the original circuit.")

### Recombining Blocks

For each block, we have a convex ensemble and a corresponding probability 
distribution. BQSKit combines these into a single `CircuitSampler` object which
is exposed to the user.

The `CircuitSampler` has 